In [3]:
import face_recognition
import cv2
import numpy as np
import os
import csv
from datetime import datetime



known_face_encodings = []
known_face_names = []

path = "faces"

print("Loading student images...")

for file in os.listdir(path):
    if file.endswith(".jpg") or file.endswith(".png"):
        image_path = os.path.join(path, file)

        image = face_recognition.load_image_file(image_path)
        encodings = face_recognition.face_encodings(image)

        if len(encodings) > 0:
            known_face_encodings.append(encodings[0])

            # File name without extension = name
            name = os.path.splitext(file)[0]
            known_face_names.append(name)

            print(f"{name} loaded successfully")
        else:
            print(f"No face found in {file}")

print("All images loaded \n")


video_capture = cv2.VideoCapture(0, cv2.CAP_DSHOW)



filename = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + ".csv"
f = open(filename, "w", newline="")
writer = csv.writer(f)
writer.writerow(["Name", "Time"])
f.flush()

marked_students = []

print("Attendance system started. Press 'q' to quit.\n")



try:
    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        # Resize for faster processing
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)

        # Convert BGR → RGB
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        # Detect faces
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        face_names = []

        for face_encoding in face_encodings:
            face_distance = face_recognition.face_distance(known_face_encodings, face_encoding)

            name = "Unknown"
            status = ""

            if len(face_distance) > 0:
                best_match_index = np.argmin(face_distance)

                if face_distance[best_match_index] < 0.5:
                    name = known_face_names[best_match_index]

            # Attendance logic
            if name != "Unknown":
                if name not in marked_students:
                    marked_students.append(name)

                    current_time = datetime.now().strftime("%H:%M:%S")
                    writer.writerow([name, current_time])
                    f.flush()

                    print(f" {name} marked present")
                    status = "Attendance Marked"
                else:
                    status = "Already Marked"

            face_names.append((name, status))

        # Draw results
        for (top, right, bottom, left), (name, status) in zip(face_locations, face_names):
            top *= 4
            right *= 4
            bottom *= 4
            left *= 4

            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)

            # Rectangle
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)

            # Name
            cv2.putText(frame, name, (left, top - 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            # Status
            cv2.putText(frame, status, (left, top - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow("Attendance System", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break



finally:
    video_capture.release()
    cv2.destroyAllWindows()
    f.close()

    print(f"\n Attendance saved: {filename}")

Loading student images...
anil loaded successfully
shiva loaded successfully
All images loaded 

Attendance system started. Press 'q' to quit.

 anil marked present
 shiva marked present

 Attendance saved: 2026-04-06_10-12-21.csv
